# Delta Lake


*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

## What is Delta Lake?

Delta Lake is a lakehouse table format that adds ACID transactions, schema enforcement, time travel, efficient updates/deletes, and streaming/batch interoperability to files stored in cloud object storage.

Under the hood, a Delta table is a directory of Parquet data files plus a `_delta_log/` directory. The log is an ordered series of JSON commits named `00000000.json`, `00000001.json`, and so on. Each commit lists the data files that were added and removed. A reader reconstructs the current snapshot by taking the newest checkpoint and replaying the later JSON commits on top of it.

A checkpoint is a Parquet file such as `0000000010.checkpoint.parquet` that holds the full file list at that version, so a reader does not replay every commit from the beginning. Delta writes a checkpoint every 10 commits by default, and the `_last_checkpoint` file points at the newest one.

Every commit is an atomic put-if-absent write. The writer uploads the log entry only if that exact path does not already exist. When two writers race, exactly one wins and the loser retries. That single mechanism is what gives Delta ACID behaviour on object storage, where there is no real rename or lock.

In Databricks, the preferred production pattern is to create governed Delta tables in Unity Catalog:

```text
catalog.schema.table
main.bronze.orders_raw
main.silver.orders
main.gold.daily_revenue
```

Real-world example: a PySpark job lands raw order events in `main.bronze.orders_raw`, cleans and deduplicates them into `main.silver.orders`, and publishes dashboard-ready aggregates into `main.gold.daily_revenue`.

A worked example of one commit. Appending two files to a table writes a file named `_delta_log/00000000000000000007.json` whose contents are, in simplified form:

```json
{"commitInfo":{"timestamp":1750000000000,"operation":"WRITE","operationParameters":{"mode":"Append"}}}
{"protocol":{"minReaderVersion":1,"minWriterVersion":2}}
{"metaData":{"id":"b0...","format":{"provider":"parquet"},"schemaString":"{...}","partitionColumns":["order_date"]}}
{"add":{"path":"order_date=2026-10-04/part-00000.parquet","size":134217728,"dataChange":true,"stats":"{\"numRecords\":1000000,\"minValues\":{\"order_id\":1},\"maxValues\":{\"order_id\":2000000},\"nullCount\":{\"order_id\":0}}"}}
{"add":{"path":"order_date=2026-10-04/part-00001.parquet","size":134000000,"dataChange":true}}
{"remove":{"path":"order_date=2026-10-03/part-00000.parquet","deletionTimestamp":1750000000000,"dataChange":false}}
```

The `stats` line is the part that pays for itself. The minimum and maximum values per indexed column let a reader skip the file without opening it.

The common mistake is to treat the table directory as ordinary Parquet. Someone points an external tool at the folder, reads the `*.parquet` files directly, and ignores `_delta_log`. That read silently includes files that were deleted or replaced, because the deletions only exist in the log. The rule is simple: read through a Delta reader, or do not read the folder at all.

## How do you create a Delta table in Databricks Unity Catalog with PySpark?

Use Unity Catalog three-part names and create schemas/tables through `spark.sql`.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("create schema if not exists main.bronze")
spark.sql("create schema if not exists main.silver")
spark.sql("create schema if not exists main.gold")

spark.sql("""
create table if not exists main.silver.orders (
  order_id bigint,
  customer_id bigint,
  order_ts timestamp,
  order_date date,
  status string,
  amount decimal(12, 2)
) using delta
partitioned by (order_date)
""")

`CREATE TABLE ... USING DELTA` writes the schema and an empty file list into version 0 of `_delta_log`. The declared types are enforced on every later write, so a string that cannot be cast to `bigint` is rejected instead of corrupting the table. That is schema enforcement in practice.

A table is managed when Unity Catalog owns both the metadata and the storage location. It is external when you give an explicit `LOCATION`, and dropping the table then leaves the files in place. Managed is the default for new work because permissions, lineage, and cleanup all follow the catalog entry.

If you are in Databricks with Unity Catalog, prefer this managed table style over writing directly to paths like `s3://.../orders`. Unity Catalog gives you centralized permissions, lineage, discovery, and table governance.

The same table can be created from the Python API, which is convenient when table definitions live in code and are reviewed like any other change.

In [ ]:
# ILLUSTRATIVE - requires delta: install to run, or read on.
from delta.tables import DeltaTable

(
    DeltaTable.createIfNotExists(spark)
    .tableName("main.silver.orders")
    .addColumn("order_id", "BIGINT")
    .addColumn("customer_id", "BIGINT")
    .addColumn("order_ts", "TIMESTAMP")
    .addColumn("order_date", "DATE")
    .addColumn("status", "STRING")
    .addColumn("amount", "DECIMAL(12, 2)")
    .partitionedBy("order_date")
    .property("delta.enableChangeDataFeed", "true")
    .execute()
)

The common mistake is to let the schema drift by creating columns as `string` and casting on every read. A cast applied at read time is a cost paid forever, and it hides bad data from the table that owns it. Decide the types once, in the `CREATE TABLE`, and fix them there when they are wrong.

## How do you write to a Delta table with PySpark?

For managed Unity Catalog tables, use `saveAsTable`, SQL inserts, or DataFrameWriterV2-style table writes where supported.

Example batch write:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

orders_df = (
    spark.read.json("s3://company-landing/orders/ds=2026-10-04/")
    .select(
        F.col("order_id").cast("bigint"),
        F.col("customer_id").cast("bigint"),
        F.col("order_ts").cast("timestamp"),
        F.to_date("order_ts").alias("order_date"),
        F.col("status").cast("string"),
        F.col("amount").cast("decimal(12,2)"),
    )
)

orders_df.write.mode("append").saveAsTable("main.silver.orders")

Every write mode has a meaning. `append` adds files. `overwrite` replaces the whole table or, with `replaceWhere`, only the rows that match a predicate. That predicate form is what makes a daily reload idempotent, because rerunning the same day replaces that day instead of duplicating it.

Delta also records `dataChange` on each added or removed file. A file removed during compaction carries `dataChange = false`, so Change Data Feed does not report it as a delete. That flag is why a downstream change reader can trust the feed across an `OPTIMIZE`.

Example partition overwrite for one day:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
(
    orders_df
    .where(F.col("order_date") == F.lit("2026-10-04"))
    .write
    .mode("overwrite")
    .option("replaceWhere", "order_date = '2026-10-04'")
    .saveAsTable("main.silver.orders")
)

Schema evolution is a write-time decision, not a table property. `mergeSchema` lets an append add new columns from the incoming data. `overwriteSchema` replaces the schema together with the data and is only valid with a full overwrite. Neither changes existing rows, so new columns arrive as nulls for old data.

```python
from pyspark.sql import functions as F

# Add a new column from the incoming data without rewriting old rows.
(
    orders_df
    .withColumn("channel", F.lit("web"))
    .write
    .option("mergeSchema", "true")
    .mode("append")
    .saveAsTable("main.silver.orders")
)
```

The deliberate path is to declare the change so every consumer sees it at the same time, or to replace the schema together with the data in a controlled overwrite.

```sql
ALTER TABLE main.silver.orders ADD COLUMNS (channel STRING);
```

```python
# overwriteSchema replaces the schema along with the data; only valid on a full overwrite.
(
    orders_df
    .write
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("main.silver.orders")
)
```

The common mistake is to leave `mergeSchema` on by default in a shared pipeline. A single malformed source file then silently adds a column to a production table, and every strict consumer breaks. Turn it on per job, review the schema diff it produces, and prefer an explicit `ALTER TABLE ... ADD COLUMNS` for planned changes.

Interview point: make writes idempotent by partition, batch id, or business key. Avoid blind append when a retry could duplicate records.

## How does Delta Lake support upserts and CDC merges?

Delta supports `MERGE INTO`, which is commonly used for CDC. In PySpark, deduplicate the source changes first, then merge into the target Delta table.

Deduplicate CDC records:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F
from pyspark.sql.window import Window

cdc = spark.table("main.bronze.orders_cdc")

window = Window.partitionBy("order_id").orderBy(
    F.col("source_lsn").desc(),
    F.col("ingested_at").desc(),
)

deduped = (
    cdc
    .withColumn("rn", F.row_number().over(window))
    .where(F.col("rn") == 1)
    .drop("rn")
)

deduped.createOrReplaceTempView("orders_cdc_deduped")

Merge changes:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
merge into main.silver.orders as target
using orders_cdc_deduped as source
on target.order_id = source.order_id
when matched and source.op = 'D' then delete
when matched then update set
  customer_id = source.customer_id,
  order_ts = source.order_ts,
  order_date = to_date(source.order_ts),
  status = source.status,
  amount = source.amount
when not matched and source.op <> 'D' then insert (
  order_id, customer_id, order_ts, order_date, status, amount
) values (
  source.order_id, source.customer_id, source.order_ts, to_date(source.order_ts), source.status, source.amount
)
""")

`MERGE` is a join followed by a rewrite. Delta reads the target files that could match the source, joins them, and rewrites only those files. A target row that matches several source rows raises an error, which is why the source must be deduplicated before the merge and not after.

On a table with deletion vectors enabled, a delete does not rewrite the whole Parquet file. Delta writes a small deletion vector that marks the row as removed, and reads apply it. The file is rewritten later by `OPTIMIZE` or `REORG TABLE ... APPLY (PURGE)`.

When the source already carries the full row, the compact form is enough.

```sql
MERGE INTO main.silver.orders AS t
USING orders_cdc_deduped AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.op = 'D' THEN DELETE
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED AND s.op <> 'D' THEN INSERT *;
```

`UPDATE SET *` and `INSERT *` copy every matching column, so the statement keeps working when a column is added later. The explicit column lists in the PySpark example are safer when the source also carries CDC metadata such as `op` and `source_lsn` that must not land in the target.

Real-world guidance: merge on stable keys, dedupe by source ordering metadata, handle deletes explicitly, and track merge row counts for observability.

The common mistake is to merge on a key that is not unique in the target. If two target rows carry the same `order_id`, the merge fails with a multiple-match error or updates both unpredictably. Enforce the key before the merge, for example with `ALTER TABLE main.silver.orders ADD CONSTRAINT orders_pk PRIMARY KEY (order_id)`, and deduplicate the source by the same key.

## How do you use Delta Lake time travel and restore?

Delta keeps table versions in the transaction log. You can query older versions for debugging, audit, or recovery.

Inspect history:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
history = spark.sql("describe history main.silver.orders")
history.select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

Each version in `DESCRIBE HISTORY` is one JSON commit, and each snapshot is reconstructed from the log plus the data files that the log still references. Time travel therefore costs almost nothing at read time. The reader lists the files as of that version and opens only those, using the same statistics as a current read.

`RESTORE` does not roll the log backwards. It commits a new version whose file list equals the old one, so history keeps growing and the restore itself is auditable.

Read an older version:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders_v42 = (
    spark.read
    .option("versionAsOf", 42)
    .table("main.silver.orders")
)

Read by timestamp:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders_yesterday = (
    spark.read
    .option("timestampAsOf", "2026-10-04 10:00:00")
    .table("main.silver.orders")
)

Restore a table to an older version:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("restore table main.silver.orders to version as of 42")

The SQL form is often what an interviewer wants to see, because it names both forms of time travel directly.

```sql
-- Snapshot reads by version and by timestamp.
SELECT * FROM main.silver.orders VERSION AS OF 42;
SELECT * FROM main.silver.orders TIMESTAMP AS OF '2026-10-04 10:00:00';

-- Restore is a new commit that points at an old snapshot.
RESTORE TABLE main.silver.orders TO VERSION AS OF 42;
RESTORE TABLE main.silver.orders TO TIMESTAMP AS OF '2026-10-04 10:00:00';
```

Do not rely on infinite time travel. Retention and `VACUUM` settings determine how far back you can safely query.

The common mistake is to assume time travel is infinite. It is bounded by two retention settings. `delta.logRetentionDuration`, 30 days by default, controls how long the commit history survives. `delta.deletedFileRetentionDuration`, one week by default, controls how long unreferenced data files survive. An old version can become unreadable while its commit is still in the log, because `VACUUM` already removed the data files it pointed at.

## What is VACUUM in Delta Lake and when should you use it?

`VACUUM` removes old data files that are no longer referenced by the current Delta table version. This controls storage cost, but it can reduce time travel history.

Example:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("vacuum main.silver.orders retain 168 hours")

The two retention settings are independent and are confused constantly. `delta.deletedFileRetentionDuration` controls how long unreferenced data files survive, and `VACUUM` deletes exactly those files once they pass that threshold. `delta.logRetentionDuration` controls how long the commit history survives. You can keep the log for 30 days and still be unable to read version 5, because the data files for it were vacuumed.

`DRY RUN` lists what would be deleted without deleting it. `RETAIN 0 HOURS` is refused unless the safety check is explicitly disabled, so start with the dry run.

```sql
-- Preview, then delete. The default threshold is 7 days.
VACUUM main.silver.orders DRY RUN;
VACUUM main.silver.orders RETAIN 168 HOURS;

-- Keep unreferenced files longer before any automatic vacuum runs.
ALTER TABLE main.silver.orders
SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '30 days');
```

Use `VACUUM` after retention requirements are clear. For example, if your team needs 30 days of rollback/audit history, do not vacuum with a 7-day retention policy.

The common mistake is to run `VACUUM RETAIN 0 HOURS` to reclaim space and then discover the incident. `VACUUM` also deletes the `_change_data` files that Change Data Feed depends on, so a downstream job that lags by more than the retention loses its change history. Pick the retention from the slowest consumer first, then vacuum.

## What is Change Data Feed in Delta Lake?

Change Data Feed (CDF) lets downstream jobs read row-level changes from a Delta table instead of scanning the full table each run.

Enable CDF:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
alter table main.silver.orders
set tblproperties (delta.enableChangeDataFeed = true)
""")

CDF stores the before and after rows of an update under `_change_data/` in the table directory, but only for operations that cannot be derived from the log. Inserts and full partition deletes are computed from the log directly, so they cost nothing extra. That is why the storage overhead is small on insert-heavy tables and larger on update-heavy ones.

CDF is off by default and only records changes made after it is enabled. Older versions have no change records, and `table_changes` on a version before the enable point raises an error.

Read changes from a starting version:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
changes = (
    spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 42)
    .table("main.silver.orders")
)

changes.select("_change_type", "_commit_version", "order_id", "status").show()

The SQL form takes versions or timestamps, inclusive on both ends, and needs only the start to read up to the latest version.

```sql
SELECT order_id, _change_type, _commit_version, order_date, amount
FROM table_changes('main.silver.orders', 42)
WHERE _change_type IN ('insert', 'update_postimage');

SELECT *
FROM table_changes('main.silver.orders', '2026-10-04 00:00:00', '2026-10-05 00:00:00');
```

The `_change_type` values are `insert`, `update_preimage`, `update_postimage`, and `delete`. An update produces two rows with the same key, so a consumer that only wants the new state filters to `insert` and `update_postimage`.

Real-world use: update a downstream `main.gold.daily_revenue` aggregate only from changed orders rather than rebuilding all history.

The common mistake is to enable CDF and then `VACUUM` the table down to a shorter window. Change records live in the table directory and follow the same retention as deleted files, so vacuuming past your slowest consumer silently removes history that consumer has not read yet. It is also wrong to expect CDF enabled yesterday to explain a change from last month, because only changes made after the enable point are recorded.

## How do Delta optimization and file compaction work?

Small files slow queries and increase metadata overhead. On Databricks, common tools include optimized writes, auto compaction, `OPTIMIZE`, and optionally Z-ordering where supported.

Optimize a table:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("optimize main.silver.orders")

Optimize a recent partition:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
optimize main.silver.orders
where order_date >= date_sub(current_date(), 7)
""")

If your workload frequently filters by `customer_id`, clustering or Z-ordering can help depending on your runtime/table features:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("optimize main.silver.orders zorder by (customer_id)")

`OPTIMIZE` reads a set of small files and rewrites them as fewer large ones. It is a rewrite, not an index. The old files are marked removed in a new commit, the new files are added, and the logical result is unchanged. The benefit is fewer files to list and open, plus tighter per-file ranges that make data skipping more effective.

Data skipping depends on the statistics in the log. Delta records `numRecords`, `minValues`, `maxValues`, and `nullCount` for the first `delta.dataSkippingNumIndexedCols` columns, 32 by default. A predicate on an indexed column lets the reader discard a file without opening it. `ZORDER` improves skipping by sorting rows so each file covers a narrow range, at the cost of rewriting the data.

Deletion vectors change the cost of deletes and updates. Without them, deleting one row rewrites every Parquet file that contains a matching row. With them, Delta writes a compact bitmap next to the file and reads apply it. The change is logical, so the removed row is still inside the file until a rewrite applies the vectors. `REORG TABLE ... APPLY (PURGE)` forces that rewrite, and `VACUUM` then removes the old file.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Auto compaction merges small files after each write; optimized writes create fewer of them.
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

spark.sql("""
ALTER TABLE main.silver.orders
SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.enableDeletionVectors'      = 'true'
)
""")

The practical answer is: compact files based on query patterns and write volume, not just on a fixed schedule.

The common mistake is to run `OPTIMIZE` on a schedule without checking file sizes first. If the average file is already close to the target, the job rewrites everything and changes almost nothing. Measure the file-size distribution with `DESCRIBE DETAIL` before scheduling anything, and compact only when the distribution justifies the rewrite.

## How do you make a Delta table Iceberg-compatible with UniForm?

Delta UniForm can generate Iceberg-compatible metadata for a Delta table, so Iceberg readers can read it. The Delta table remains the source of truth.

Create a Unity Catalog Delta table with Iceberg compatibility enabled:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
create table if not exists main.silver.orders_uniform (
  order_id bigint,
  customer_id bigint,
  order_ts timestamp,
  order_date date,
  amount decimal(12, 2)
) using delta
tblproperties (
  'delta.enableIcebergCompatV2' = 'true',
  'delta.universalFormat.enabledFormats' = 'iceberg'
)
partitioned by (order_date)
""")

UniForm writes Iceberg metadata next to the Delta log on every commit, in the table's `metadata/` directory, while Parquet remains the only physical data format. An Iceberg reader therefore reads the same files as the Delta reader, with no copy and no second table.

It is a metadata bridge, not a merge of the two formats. The Delta table stays the source of truth, the Iceberg metadata is generated from it, and writes still go through the Delta writer. Enabling it upgrades the table protocol, so older readers that do not understand the compatibility features can no longer read the table.

Enable UniForm on an existing table:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
alter table main.silver.orders_uniform
set tblproperties (
  'delta.enableIcebergCompatV2' = 'true',
  'delta.universalFormat.enabledFormats' = 'iceberg'
)
""")

Important: treat Iceberg access to UniForm tables as read-oriented unless your platform explicitly supports writes through multiple formats. In most interview scenarios, Delta owns writes and external Iceberg clients read.

The common mistake is to advertise a UniForm table as writable from every engine. Outside Databricks, Iceberg clients on a UniForm table are read-only. Plan the write path as Delta only, and treat Iceberg access as a read contract for external engines. Also check the writer version the compatibility properties require, because setting them on a table whose writers do not support them fails at commit time.

## Delta Lake vs Iceberg: when would you choose each?

Choose Delta Lake when your platform is Databricks-centered and you want tight integration with Unity Catalog, Delta Live Tables, Databricks SQL, Auto Loader, Change Data Feed, and Databricks optimization features.

Choose Iceberg when you need broad multi-engine interoperability as the primary requirement, especially when Spark, Trino, Flink, Snowflake, and other engines need to share open table metadata directly.

The formats differ first in their metadata model. Delta keeps an ordered JSON log of add and remove actions with periodic Parquet checkpoints. Iceberg keeps a manifest list that points at manifest files, which point at data files, with snapshots on top. Hudi keeps a timeline of actions and chooses copy-on-write or merge-on-read per table. All three solve ACID on object storage. They differ in what the metadata is optimized for and which engines read it natively.

Example Delta fit:

```text
Databricks PySpark writes main.silver.orders
Databricks SQL dashboards query main.gold.daily_revenue
Unity Catalog owns permissions and lineage
Change Data Feed powers downstream aggregates
```

Example Iceberg fit:

```text
PySpark writes main.analytics.orders as Iceberg
Snowflake reads through Unity Catalog Iceberg REST
Trino reads through Iceberg REST
Multiple engines need the same open table contract
```

A short comparison of the table formats, including the one that is not a table format.

| Format | Metadata | Best fit |
| --- | --- | --- |
| Delta Lake | `_delta_log` JSON commits plus checkpoints | Databricks-centric lakehouse, streaming merges, CDF |
| Apache Iceberg | Manifest list, manifests, snapshots | Multi-engine reads, engine-neutral catalogs |
| Apache Hudi | Timeline of actions, copy-on-write or merge-on-read | Record-level upserts on HDFS-style lakes |
| Plain Parquet | Per-file footer only | Immutable data and simple overwrite loads |

Plain Parquet is not a table format. There is no transaction across files, no delete, no time travel, and no schema enforced across the directory. A schema change means every downstream reader needs a new contract. Delta, Iceberg, and Hudi each add the missing metadata layer on top of the same Parquet files.

There is overlap. Delta UniForm can expose Delta tables to Iceberg readers, while native Iceberg is often cleaner when Iceberg is the primary table contract from day one.

The common mistake is to compare the formats as if one were strictly faster. They store the same Parquet files, so the differences are in metadata, deletion mechanics, and ecosystem support. The deciding questions are which engines must read the table, how often row-level deletes happen, and which catalog the organization already runs. Do not migrate a working table for a benchmark you have not run.

## Real world use cases for Delta Lake

Read each case as an answer template. The shape is always the same: what arrives, which Delta feature does the hard part, and what the alternative would cost. The detail that makes an answer credible is the mechanism, not the product name.

### 1. Medallion lakehouse on Databricks

Bronze holds raw events, silver holds cleaned and deduplicated entities, gold holds aggregates. Silver and gold are Delta tables maintained with `MERGE` plus scheduled `OPTIMIZE`.

In [ ]:
# ILLUSTRATIVE - requires delta: install to run, or read on.
from delta.tables import DeltaTable

# Upsert yesterday's changed orders into silver
(
    DeltaTable.forName(spark, "main.silver.orders")
    .alias("t")
    .merge(
        spark.read.table("main.bronze.order_events").alias("s"),
        "t.order_id = s.order_id",
    )
    .whenMatchedUpdateAll(condition="s.updated_at > t.updated_at")
    .whenNotMatchedInsertAll()
    .execute()
)

# Compact and co-locate for the most common filter
spark.sql("OPTIMIZE main.silver.orders ZORDER BY (customer_id)")

Why Delta: ACID merges make incremental silver and gold layers reliable, and time travel gives reproducible reporting without a separate history store.

### 2. Kafka CDC into Delta with exactly-once streaming

A Structured Streaming job consumes Debezium events and merges each micro-batch into Delta. Checkpointing gives exactly-once processing, and `MERGE` makes the write idempotent even if a batch is replayed.

In [ ]:
# ILLUSTRATIVE - requires delta: install to run, or read on.
from delta.tables import DeltaTable

def upsert_to_delta(micro_batch_df, batch_id):
    (
        DeltaTable.forName(spark, "main.silver.customers")
        .alias("t")
        .merge(micro_batch_df.alias("s"), "t.customer_id = s.customer_id")
        .whenMatchedDelete(condition="s.op = 'd'")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

(
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092")
    .option("subscribe", "cdc.public.customers")
    .load()
    .selectExpr("CAST(value AS STRING) AS json")
    .writeStream
    .foreachBatch(upsert_to_delta)
    .option("checkpointLocation", "s3://acme-checkpoints/customers/")
    .trigger(availableNow=True)
    .start()
)

Why Delta: CDC deletes and updates map directly to `MERGE`, and replaying a batch does not duplicate rows.

### 3. Change Data Feed for downstream aggregates

Gold aggregates only need changed rows, not full rebuilds. Enable CDF and read the change stream to update a metrics table incrementally.

```sql
ALTER TABLE main.silver.orders
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

SELECT order_id, _change_type, order_date, amount
FROM table_changes('main.silver.orders', '2026-10-04T00:00:00', '2026-10-05T00:00:00')
WHERE _change_type IN ('insert', 'update_postimage');
```

Why Delta: incremental propagation keeps the end-to-end SLA short without recomputing the whole warehouse.

### 4. Regulatory reporting, audit, and right-to-be-forgotten

A bank keeps every version of a transactions table for audit, then must also delete one customer's data on request.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# point-in-time report as of the close
spark.read.option("versionAsOf", 42).table("main.gold.fct_transactions")

# delete, then physically remove files that are no longer referenced
spark.sql("DELETE FROM main.gold.fct_transactions WHERE customer_id = 'C-12345'")
spark.sql("VACUUM main.gold.fct_transactions RETAIN 168 HOURS")

Why Delta: time travel answers audit questions, and delete plus `VACUUM` removes data for good - but the retention window must stay shorter than the time-travel window your readers depend on.

### 5. One table for streaming and batch

A pipeline ingests continuously and also re-reads history for backfills. Because both paths use the same Delta table, they share one schema, one history, and one permission model.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# batch backfill from a known-good version
backfill = spark.read.option("versionAsOf", 40).table("main.silver.orders")

# recovery: undo a bad publish, then reprocess
spark.sql("RESTORE TABLE main.silver.orders TO VERSION AS OF 40")

Why Delta: one storage contract removes the classic streaming-table versus batch-table split, and `RESTORE` makes a bad deployment reversible in seconds.

Across the cases the recurring mechanics are the same four. The transaction log supplies atomicity and snapshot reads. `MERGE` supplies row-level change into an immutable file store. Time travel and `RESTORE` supply audit and recovery. `OPTIMIZE`, clustering, and `VACUUM` supply read performance and retention. If you can name which of the four a scenario needs, the design follows.

## How do you tune Delta tables for performance?

Tuning a Delta table starts with a diagnosis, not a setting. Decide which of three problems you have: the query reads too much data, the write rewrites too much data, or the table carries too much metadata. Layout changes address the first, deletion-vector and merge design address the second, and compaction addresses the third. Name the problem before changing anything.

File sizing is the first lever. Target roughly 100 MB to 1 GB per file: large enough that a scan does not pay per-file overhead, small enough that the file count still gives the cluster something to parallelize. Streaming appends create thousands of tiny files unless optimized writes or auto compaction merge them, and a scheduled `OPTIMIZE` clears the backlog. The opposite failure is a handful of enormous files, which cap parallelism and make every rewrite expensive.

Data skipping is the second lever, and it depends on statistics. Delta records `numRecords`, `minValues`, `maxValues`, and `nullCount` for the first `delta.dataSkippingNumIndexedCols` columns, 32 by default, and stores them in the log and in checkpoints. At read time the engine compares the predicate with each file's range and skips files that cannot match. Statistics only help when the predicate column is indexed and the ranges are tight, which is exactly what `ZORDER` and liquid clustering produce.

Liquid clustering is the recommended layout for new tables. Declare the columns that queries filter on with `CLUSTER BY`, then run `OPTIMIZE`; clustering is incremental, so only the data that must move is rewritten. Up to four clustering columns are allowed. Partitioning is narrower: use it only for a genuinely low-cardinality column that almost every query filters, and never combine it with clustering or `ZORDER`. Changing clustering keys does not rewrite history until `OPTIMIZE ... FULL`.

Deletion vectors decide how expensive updates and deletes are. Without them, changing one row rewrites every file that holds a matching row. With `delta.enableDeletionVectors`, Delta records a bitmap and leaves the Parquet file in place, so the write is cheap and the work is deferred. Reads pay a small cost to apply the vector. The soft-deleted rows are physically removed only when `OPTIMIZE` or `REORG TABLE ... APPLY (PURGE)` rewrites the file, and only `VACUUM` removes the old file.

`OPTIMIZE` costs real money because it reads and rewrites data. On a clustered table the incremental run is cheap, while `OPTIMIZE ... FULL` on a large table can run for hours. Compare the benefit with the cost. A scan that skips 95 percent of files usually saves more than the rewrite costs, while compacting files that are already near the target saves nothing. This is the general rule: fix layout before adding cluster size, because a bigger cluster over a bad layout just spends more to read the same bytes.

Layout and joins interact. Adaptive query execution can turn a sort-merge join into a broadcast join when one side is small after filtering, and a `BROADCAST` hint forces it when statistics are stale. Clustering a large fact table on the join key reduces the bytes that reach the join, but it does not fix skew. If one key holds a large share of rows, one task still does most of the work. The fixes there are AQE skew handling, a `SKEW` hint, or salting the key, and none of them are a table-layout change.

```sql
-- Inspect size and layout before changing anything.
DESCRIBE DETAIL main.silver.orders;
DESCRIBE HISTORY main.silver.orders LIMIT 10;

-- Compact small files; on a clustered table this also clusters incrementally.
OPTIMIZE main.silver.orders;
OPTIMIZE main.silver.orders WHERE order_date >= current_date() - INTERVAL 7 DAYS;
OPTIMIZE main.silver.orders FULL;

-- Layout choice one: liquid clustering, up to four columns, incremental.
ALTER TABLE main.silver.orders CLUSTER BY (order_date, customer_id);

-- Layout choice two: ZORDER, valid only on a table that is not clustered.
-- OPTIMIZE main.silver.orders ZORDER BY (customer_id);

-- Keep statistics wide enough for the predicates you actually use.
ALTER TABLE main.silver.orders
SET TBLPROPERTIES ('delta.dataSkippingNumIndexedCols' = '32');

-- Apply soft deletes so removed rows leave the current files.
REORG TABLE main.silver.orders APPLY (PURGE);
```

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Measure the file-size distribution from the table detail, then decide.
detail = DeltaTable.forName(spark, "main.silver.orders").detail().collect()[0]
num_files = detail["numFiles"]
size_bytes = detail["sizeInBytes"]
avg_mb = size_bytes / num_files / (1024 ** 2)

print(f"files={num_files} avg_file_mb={avg_mb:.1f}")

# Compact when the average is far below the target; leave it alone otherwise.
if avg_mb < 100:
    spark.sql("OPTIMIZE main.silver.orders")


The common mistake is to tune in the wrong order. People add workers, or raise a shuffle setting, before checking how many files exist and whether the predicate columns are indexed. Both questions have cheap evidence: `DESCRIBE DETAIL` for the file count and average size, and `DESCRIBE HISTORY` plus the commit statistics for which files a predicate can skip. Spend the cluster budget only after the layout evidence says it will help.

## Delta exercises

These exercises use only the Python standard library and model the decisions in the sections above. Each one states the task, gives a hint, names the expected result, and then prints and asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The numbers are illustrative. The point of each exercise is to know which quantity to compute and which direction a change moves it: deduplication before a merge decides which branch each row takes, the target file count follows from table size and target file size, and a vacuum is safe only when its retention is shorter than the history consumers still need.

### Try it 1. Plan a MERGE keyed on updated_at

**Task.** Staged rows are duplicated per `order_id`. Keep only the row with the greatest `updated_at`, then classify each survivor against the target and count the updates, inserts, and deletes the merge performs. A staged row with `op == "D"` is a delete when the key exists in the target, and is ignored otherwise.

**Hint.** Sort by `updated_at` descending and keep the first row per key before comparing with the target.

**Expected result.** 4 deduplicated staged rows, 1 update, 1 insert, 1 delete, and 1 ignored delete.

In [ ]:
staged = [
    {"order_id": 1, "updated_at": 100, "op": "U", "amount": 10.0},
    {"order_id": 1, "updated_at": 120, "op": "U", "amount": 12.0},
    {"order_id": 2, "updated_at": 110, "op": "U", "amount": 20.0},
    {"order_id": 3, "updated_at": 90, "op": "D"},
    {"order_id": 5, "updated_at": 80, "op": "D"},
]
target = {1: {"amount": 10.0}, 3: {"amount": 30.0}}

latest = {}
for row in sorted(staged, key=lambda r: r["updated_at"], reverse=True):
    latest.setdefault(row["order_id"], row)
deduped = list(latest.values())

updates = [r for r in deduped if r["op"] != "D" and r["order_id"] in target]
inserts = [r for r in deduped if r["op"] != "D" and r["order_id"] not in target]
deletes = [r for r in deduped if r["op"] == "D" and r["order_id"] in target]
ignored = [r for r in deduped if r["op"] == "D" and r["order_id"] not in target]

print("deduped:", len(deduped))
print("updates:", [r["order_id"] for r in updates])
print("inserts:", [r["order_id"] for r in inserts])
print("deletes:", [r["order_id"] for r in deletes])
print("ignored deletes:", [r["order_id"] for r in ignored])

assert len(deduped) == 4
assert [r["order_id"] for r in updates] == [1]
assert updates[0]["amount"] == 12.0
assert [r["order_id"] for r in inserts] == [2]
assert [r["order_id"] for r in deletes] == [3]
assert [r["order_id"] for r in ignored] == [5]


### Try it 2. Compute target file count from table size

**Task.** A table is 3 TB and the target file size is 256 MB. Compute the target file count, the current average file size given 120,000 files, and whether compaction is needed, treating an average below half the target as too small.

**Hint.** One TB is `1024 ** 2` MB, and compaction is needed when the average is below `target_mb / 2`.

**Expected result.** 12,288 target files, a current average of 26.2 MB, and compaction is needed.

In [ ]:
import math

MB_PER_TB = 1024 ** 2
table_tb = 3.0
target_mb = 256
current_files = 120_000

table_mb = table_tb * MB_PER_TB
target_files = math.ceil(table_mb / target_mb)
current_avg_mb = table_mb / current_files
needs_compaction = current_avg_mb < target_mb / 2

print(f"table = {table_mb:.0f} MB")
print(f"target files at {target_mb} MB = {target_files}")
print(f"current average file = {current_avg_mb:.1f} MB")
print("needs compaction:", needs_compaction)

assert target_files == 12288
assert round(current_avg_mb, 1) == 26.2
assert needs_compaction is True


### Try it 3. Decide partitioning versus liquid clustering

**Task.** Given table size and per-column `cardinality` and `filter_rate`, choose a layout. Partition only when the table is at least 200 GB and some column is filtered by at least 80 percent of queries with cardinality at most 1,000. Leave tables under 50 GB unchanged. Otherwise cluster on the two most-filtered columns.

**Hint.** Check the partition condition first, then sort the columns by `filter_rate` for the clustering keys.

**Expected result.** The 1 TB events table partitions by `event_date`, the 1 TB clicks table clusters on `(customer_id, device_id)`, and the 10 GB table is left as plain Delta.

In [ ]:
def choose_layout(size_gb, columns):
    partition_cols = [
        name for name, stats in columns.items()
        if stats["filter_rate"] >= 0.8 and stats["cardinality"] <= 1_000
    ]
    if size_gb >= 200 and partition_cols:
        return "partition by " + partition_cols[0]
    if size_gb < 50:
        return "leave as plain Delta"
    keys = sorted(columns, key=lambda n: columns[n]["filter_rate"], reverse=True)[:2]
    return "cluster by (" + ", ".join(keys) + ")"


events = {
    "event_date": {"cardinality": 365, "filter_rate": 1.0},
    "customer_id": {"cardinality": 1_000_000, "filter_rate": 0.6},
}
clicks = {
    "customer_id": {"cardinality": 5_000_000, "filter_rate": 0.95},
    "device_id": {"cardinality": 2_000_000, "filter_rate": 0.9},
    "country": {"cardinality": 40, "filter_rate": 0.05},
}
small = {
    "event_date": {"cardinality": 365, "filter_rate": 1.0},
    "customer_id": {"cardinality": 1_000_000, "filter_rate": 0.6},
}

plans = {
    "events": choose_layout(1000, events),
    "clicks": choose_layout(1000, clicks),
    "small": choose_layout(10, small),
}
for name, plan in plans.items():
    print(f"{name}: {plan}")

assert plans["events"] == "partition by event_date"
assert plans["clicks"] == "cluster by (customer_id, device_id)"
assert plans["small"] == "leave as plain Delta"


### Try it 4. Estimate OPTIMIZE benefit from a file-size distribution

**Task.** Given the sizes of the files in a table, compute the file count and average size before compaction, the file count after compacting to a 256 MB target, the percentage reduction in file count, and whether to run `OPTIMIZE`, treating an average below 64 MB as worth it.

**Hint.** `files_after = ceil(total_mb / target_mb)` and reduction is `(before - after) / before`.

**Expected result.** 40 files at 10.0 MB average, 2 files after compaction, a 95.0 percent reduction, and `OPTIMIZE` is worth running.

In [ ]:
import math

file_sizes_mb = [10.0] * 40
target_mb = 256
compact_threshold_mb = 64

files_before = len(file_sizes_mb)
total_mb = sum(file_sizes_mb)
avg_before = total_mb / files_before
files_after = math.ceil(total_mb / target_mb)
reduction_pct = (files_before - files_after) / files_before * 100
should_optimize = avg_before < compact_threshold_mb

print(f"total = {total_mb:.0f} MB in {files_before} files (avg {avg_before:.1f} MB)")
print(f"after OPTIMIZE: {files_after} files")
print(f"file-count reduction: {reduction_pct:.1f}%")
print("run OPTIMIZE:", should_optimize)

assert files_before == 40
assert avg_before == 10.0
assert files_after == 2
assert round(reduction_pct, 1) == 95.0
assert should_optimize is True


### Try it 5. Compute time-travel retention versus VACUUM safety

**Task.** A table must keep 30 days of time travel. The log retention is 45 days, the deleted-file retention is 7 days, and a job proposes `VACUUM ... RETAIN 168 HOURS`. Convert the retention windows to days, decide whether the vacuum is safe, compute the minimum retention in hours that keeps 30 days, and say whether the log retention is the binding limit.

**Hint.** Hours divided by 24 gives days, and the vacuum is safe only when it is at least the required history.

**Expected result.** The vacuum allows 7 days, it is not safe for 30 days of time travel, the minimum is 720 hours, and the log retention is not the binding limit.

In [ ]:
required_days = 30
log_retention_days = 45
deleted_file_retention_days = 7
vacuum_retain_hours = 168

vacuum_days = vacuum_retain_hours / 24
safe = vacuum_days >= required_days
minimum_hours = required_days * 24
log_is_binding = log_retention_days < required_days

print(f"log retention: {log_retention_days} days")
print(f"deleted-file retention: {deleted_file_retention_days} days")
print(f"proposed vacuum retention: {vacuum_retain_hours} h = {vacuum_days:.0f} days")
print("vacuum safe for 30 days of time travel:", safe)
print(f"minimum vacuum retention: {minimum_hours} hours")
print("log retention is the binding limit:", log_is_binding)

assert vacuum_days == 7.0
assert safe is False
assert minimum_hours == 720
assert log_is_binding is False


### Try it 6. Detect CDF change types for a row update

**Task.** Given the previous and current rows for a set of keys, return the change rows and their `_change_type`. A new key emits `insert`, a removed key emits `delete`, a changed key emits `update_preimage` then `update_postimage`, and an unchanged key emits nothing.

**Hint.** Compare the rows by key and emit the preimage before the postimage.

**Expected result.** Four change rows: an update on key 1, an insert on key 2, and a delete on key 3. Key 4 is unchanged and emits nothing.

In [ ]:
before = {
    1: {"status": "open", "amount": 10.0},
    3: {"status": "shipped", "amount": 30.0},
    4: {"status": "open", "amount": 40.0},
}
after = {
    1: {"status": "closed", "amount": 12.0},
    2: {"status": "open", "amount": 20.0},
    4: {"status": "open", "amount": 40.0},
}

changes = []
for key in sorted(set(before) | set(after)):
    old, new = before.get(key), after.get(key)
    if old is None:
        changes.append((key, "insert"))
    elif new is None:
        changes.append((key, "delete"))
    elif old != new:
        changes.append((key, "update_preimage"))
        changes.append((key, "update_postimage"))

for key, change_type in changes:
    print(f"order_id={key} {change_type}")

assert changes == [
    (1, "update_preimage"),
    (1, "update_postimage"),
    (2, "insert"),
    (3, "delete"),
]
assert len(changes) == 4


### Try it 7. Count files skipped by statistics

**Task.** Each file records `rows` and a `min` and `max` for `order_date`. For the predicate `order_date = '2026-10-04'`, count the files read, the files skipped, the rows skipped, and the percentage of files read.

**Hint.** A file can match when the value falls inside its `min` and `max`.

**Expected result.** 2 files read, 3 files skipped, 3,000,000 rows skipped, and 40 percent of files read.

In [ ]:
files = [
    {"path": "f1", "rows": 1_000_000, "min": "2026-10-01", "max": "2026-10-02"},
    {"path": "f2", "rows": 1_000_000, "min": "2026-10-03", "max": "2026-10-03"},
    {"path": "f3", "rows": 1_000_000, "min": "2026-10-04", "max": "2026-10-04"},
    {"path": "f4", "rows": 1_000_000, "min": "2026-10-05", "max": "2026-10-06"},
    {"path": "f5", "rows": 1_000_000, "min": "2026-10-04", "max": "2026-10-04"},
]
predicate = "2026-10-04"


def can_match(file, value):
    return file["min"] <= value <= file["max"]


read = [f for f in files if can_match(f, predicate)]
skipped = [f for f in files if not can_match(f, predicate)]

print("files read:", [f["path"] for f in read])
print("files skipped:", [f["path"] for f in skipped])
print("rows skipped:", sum(f["rows"] for f in skipped))
print(f"files read pct: {len(read) / len(files) * 100:.0f}%")

assert [f["path"] for f in read] == ["f3", "f5"]
assert [f["path"] for f in skipped] == ["f1", "f2", "f4"]
assert sum(f["rows"] for f in skipped) == 3_000_000
assert round(len(read) / len(files) * 100) == 40


### Try it 8. Pick a table format from requirements

**Task.** For each scenario choose `parquet`, `iceberg`, `delta`, or `hudi`. Choose `parquet` for immutable data that is only overwritten wholesale. Choose `iceberg` when more than one engine must read the table and no single platform owns it. Choose `delta` when the platform is Databricks-centric and needs streaming merges, CDF, and time travel. Choose `hudi` when record-level upserts with an incremental pull are required on an HDFS-style lake.

**Hint.** Apply the rules in order, and remember that `parquet` is not a table format.

**Expected result.** `parquet`, `iceberg`, `delta`, `hudi` in scenario order.

In [ ]:
def choose_format(req):
    if req["immutable"] and req["write_mode"] == "overwrite":
        return "parquet"
    if req["multi_engine"] and not req["single_platform"]:
        return "iceberg"
    if req["single_platform"] == "databricks":
        return "delta"
    if req["record_level_upserts"] and req["lake_style"] == "hdfs":
        return "hudi"
    return "delta"


scenarios = {
    "daily_extract": {
        "immutable": True, "write_mode": "overwrite", "multi_engine": False,
        "single_platform": None, "record_level_upserts": False, "lake_style": "object",
    },
    "shared_analytics": {
        "immutable": False, "write_mode": "append", "multi_engine": True,
        "single_platform": False, "record_level_upserts": False, "lake_style": "object",
    },
    "databricks_lakehouse": {
        "immutable": False, "write_mode": "merge", "multi_engine": False,
        "single_platform": "databricks", "record_level_upserts": True, "lake_style": "object",
    },
    "hdfs_upserts": {
        "immutable": False, "write_mode": "merge", "multi_engine": False,
        "single_platform": None, "record_level_upserts": True, "lake_style": "hdfs",
    },
}

picks = {name: choose_format(req) for name, req in scenarios.items()}
for name, fmt in picks.items():
    print(f"{name}: {fmt}")

assert picks == {
    "daily_extract": "parquet",
    "shared_analytics": "iceberg",
    "databricks_lakehouse": "delta",
    "hdfs_upserts": "hudi",
}
